# Radar signal classification: run an experiment suite on Colab

1. Runtime > Change runtime type > **GPU** (L4 or A100).
2. Add these in **Secrets** (key icon on the left) and enable notebook access: `WANDB_API_KEY`, `KAGGLE_USERNAME`, `KAGGLE_KEY` (from kaggle.com > Settings > API > Create New Token).
3. Run the cells. Runs are saved to Google Drive (`MyDrive/radar_runs`), so if the runtime disconnects, run all cells again and the suite continues where it stopped.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# `!python` subprocesses cannot read Colab Secrets, so export them as environment variables.
import os
from google.colab import userdata

for name in ("WANDB_API_KEY", "KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
    try:
        os.environ[name] = userdata.get(name).strip()
        print(f"{name}: loaded")
    except Exception as e:  # missing secret or notebook access off
        print(f"{name}: not available ({type(e).__name__})")


In [ ]:
BRANCH = 'experiments/ablation-baselines-wandb'
!rm -rf /content/repo && git clone -q -b $BRANCH https://github.com/vinhhuy15/Radar-Signal-Classification-DL /content/repo
!cd /content/repo && git log --oneline -1 && pip install -q -e '.[onnx]' && nvidia-smi -L

Baselines at 40 epochs, 3 seeds. `rf_spectronet` is skipped here because the identical run is `A5_full` in the Kaggle ablation.

In [ ]:
!cd /content/repo && python scripts/colab_run.py baselines --only mobilenetv3_small_w0.2 shufflenetv2_tiny resnet_tiny --set train.epochs=40

In [ ]:
!cat /content/results/baselines.md